In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import pandas as pd
import spacy
import re
import time
import numpy as np
import time

import ast

In [175]:
from spacy.matcher import PhraseMatcher
from spacy.matcher import Matcher
import re

In [176]:
# Model data
model = {'en': spacy.load("en_core_web_sm"), 'de': spacy.load("de_core_news_sm")}

In [177]:
#function to split string in the row by " | "
def split_form(row):
    row_split = row.split(" | ")
    return row_split

In [178]:
#function to split words and sentences into two different dataframes (currently doesn't used)
def split_words_sentences_short(df):
    list_word = []
    list_sentence = []

    list_word_type_word = []
    list_word_type_sentence = []

    list_category_word = []
    list_category_sentence = []

    list_primary_subcategory_word = []
    list_primary_subcategory_sentence = []

    list_alternatives_word = []
    list_alternatives_sentence = []

    for index, row in df.iterrows():
        word = row["Lemma"]
        word_list = word.split() # Split `a_string` by whitespace.
        number_of_words = len(word_list)
        if number_of_words == 1:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_alternatives_word.append(row["Alt_split"])
        
        
        else:
            list_sentence.append(word)
            list_word_type_sentence.append(row["Word_Type"])
            list_category_sentence.append(row["Category"])
            list_primary_subcategory_sentence.append(row["Primary_subcategory"])
            list_alternatives_sentence.append(row["Alt_split"])
        
    df_word = pd.DataFrame(list(zip(list_word, list_word_type_word, list_category_word, list_primary_subcategory_word, list_alternatives_word)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Alt_split"])
    df_sentence = pd.DataFrame(list(zip(list_sentence, list_word_type_sentence, list_category_sentence, list_primary_subcategory_sentence, list_alternatives_sentence)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Alt_split"])
    
    return df_word, df_sentence
        
    


In [179]:
#function to split words and sentences into two different dataframes for singular/plural dataframes
def split_words_sentences(df):
    list_word = []
    list_sentence = []

    list_word_type_word = []
    list_word_type_sentence = []

    list_category_word = []
    list_category_sentence = []

    list_primary_subcategory_word = []
    list_primary_subcategory_sentence = []

    list_sg_all_word = []
    list_sg_all_sentence = []
    
    list_pl_word = []
    list_pl_sentence = []

    list_pl_all_word = []
    list_pl_all_sentence = []

    list_alternatives_word = []
    list_alternatives_sentence = []
    
    for index, row in df.iterrows():
        word = row["Lemma"]
        word_list = word.split() # Split `a_string` by whitespace.
        number_of_words = len(word_list)
        if number_of_words == 1:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_sg_all_word.append(row["Sg_all_split"])
            list_pl_word.append(row["Pl_all"])
            list_pl_all_word.append(row["Pl_all_split"])
            list_alternatives_word.append(row["Alt_split"])
        
        
        else:
            list_sentence.append(word)
            list_word_type_sentence.append(row["Word_Type"])
            list_category_sentence.append(row["Category"])
            list_primary_subcategory_sentence.append(row["Primary_subcategory"])
            list_sg_all_sentence.append(row["Sg_all_split"])
            list_pl_sentence.append(row["Pl_all"])
            list_pl_all_sentence.append(row["Pl_all_split"])
            list_alternatives_sentence.append(row["Alt_split"])
        
    df_word = pd.DataFrame(list(zip(list_word, list_word_type_word, list_category_word, list_primary_subcategory_word, list_sg_all_word, list_pl_word, list_pl_all_word, list_alternatives_word)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])
    df_sentence = pd.DataFrame(list(zip(list_sentence, list_word_type_sentence, list_category_sentence, list_primary_subcategory_sentence, list_sg_all_sentence, list_pl_sentence, list_pl_all_sentence, list_alternatives_sentence)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])
    
    return df_word, df_sentence
        
    


In [180]:
# read csv file
df_rules = pd.read_csv("rules_23APR_English_h.csv")

In [181]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1727 entries, 0 to 1726
Data columns (total 15 columns):
Lemma                     1727 non-null object
Word_Type                 1727 non-null object
Priority                  801 non-null object
Category                  1727 non-null object
Primary_subcategory       1726 non-null object
Secondary_subcategory     611 non-null object
Alt_Field                 476 non-null object
Alt_Sg_Replacement        997 non-null object
Alt_Sg_people_first       160 non-null object
Alt_Sg_reframed           302 non-null object
Alt_Pl_collective_noun    53 non-null object
Alt_Pl                    291 non-null object
Alt_Pl_people_first       87 non-null object
Alt_Pl_reframed           53 non-null object
3rd_party_alternatives    2 non-null object
dtypes: object(15)
memory usage: 202.5+ KB


### Drop duplicates

In [182]:
df_rules = df_rules.drop_duplicates()

In [191]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1726 entries, 0 to 1726
Data columns (total 15 columns):
Lemma                     1726 non-null object
Word_Type                 1726 non-null object
Priority                  801 non-null object
Category                  1726 non-null object
Primary_subcategory       1725 non-null object
Secondary_subcategory     611 non-null object
Alt_Field                 475 non-null object
Alt_Sg_Replacement        997 non-null object
Alt_Sg_people_first       160 non-null object
Alt_Sg_reframed           302 non-null object
Alt_Pl_collective_noun    53 non-null object
Alt_Pl                    291 non-null object
Alt_Pl_people_first       87 non-null object
Alt_Pl_reframed           53 non-null object
3rd_party_alternatives    2 non-null object
dtypes: object(15)
memory usage: 215.8+ KB


### Check if subcategory name is missing and set the parent category name to subcategory if the subategory name is missing

In [193]:
#check missing values in Primary subcategory
df_rules[df_rules['Primary_subcategory'].isna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_people_first,Alt_Pl_reframed,3rd_party_alternatives
1025,impact,v,NaN,inclusive,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [194]:
#fill the missing values in "primary subcategory" by values from "Category"
df_rules.Primary_subcategory.fillna(df_rules.Category, inplace=True)

In [196]:
#check if the values are filled
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1726 entries, 0 to 1726
Data columns (total 15 columns):
Lemma                     1726 non-null object
Word_Type                 1726 non-null object
Priority                  801 non-null object
Category                  1726 non-null object
Primary_subcategory       1726 non-null object
Secondary_subcategory     611 non-null object
Alt_Field                 475 non-null object
Alt_Sg_Replacement        997 non-null object
Alt_Sg_people_first       160 non-null object
Alt_Sg_reframed           302 non-null object
Alt_Pl_collective_noun    53 non-null object
Alt_Pl                    291 non-null object
Alt_Pl_people_first       87 non-null object
Alt_Pl_reframed           53 non-null object
3rd_party_alternatives    2 non-null object
dtypes: object(15)
memory usage: 215.8+ KB


## Alternatives Single

### Single click+rephrase. Important order of columns with alternatives

In [99]:
#order for single form of alternatives provided by Tracey
df_rules["Sg_all"]= df_rules.loc[:, ("Alt_Sg_Replacement", "Alt_Field", "Alt_Pl_collective_noun", "Alt_Pl", "Alt_Sg_people_first", "Alt_Sg_reframed")].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)

In [100]:
# split alternatives in the row by " | " applying function split_form to convert string alternatives into array
df_rules["Sg_all_split"]=df_rules["Sg_all"].map(split_form)

## Alternatives Plural

### Single click + rephrase. Important order of columns with alternatives

In [101]:
#order for plural form of alternatives provided by Tracey
df_rules["Pl_all"]= df_rules.loc[:, ("Alt_Pl", "Alt_Pl_collective_noun", "Alt_Field", "Alt_Pl_people_first")].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)

In [102]:
# split alternatives in the row by " | " applying function split_form
df_rules["Pl_all_split"]=df_rules["Pl_all"].map(split_form)

### Merge alternatives into one column

In [103]:
source_col_loc = df_rules.columns.get_loc('Alt_Field') # column position starts from 0

In [104]:
#merge alternatives in one column by sign  " | "
df_rules['alternatives']= df_rules.iloc[:,source_col_loc:source_col_loc+7].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)

In [105]:
# split alternatives in the row by " | " applying function split_form
df_rules["Alt_split"]=df_rules["alternatives"].map(split_form)

In [106]:
df_rules.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_people_first,Alt_Sg_reframed,...,Alt_Pl,Alt_Pl_people_first,Alt_Pl_reframed,3rd_party_alternatives,Sg_all,Sg_all_split,Pl_all,Pl_all_split,alternatives,Alt_split
0,(x)aaS,acr,NaN,style,abbreviation,marketing,NaN,something as a service | something as a servic...,NaN,NaN,...,NaN,NaN,NaN,NaN,something as a service | something as a servic...,"[something as a service, something as a servic...",,[],something as a service | something as a servic...,"[something as a service, something as a servic..."
1,ACCT,abbr,NaN,style,abbreviation,finance,NaN,account,NaN,NaN,...,NaN,NaN,NaN,NaN,account,[account],,[],account,[account]
2,ADD,abbr,11-02-2022,style,abbreviation,misused_medical_term,NaN,attention deficit disorder | attention deficit...,NaN,easily bored | easily distracted | having a po...,...,NaN,NaN,NaN,NaN,attention deficit disorder | attention deficit...,"[attention deficit disorder, attention deficit...",,[],attention deficit disorder | attention deficit...,"[attention deficit disorder, attention deficit..."
3,ADHD,abbr,11-02-2022,style,abbreviation,misused_medical_term,NaN,attention deficit hyperactivity disorder | att...,person with attention deficit hyperactivity di...,fidgety | restless | easily bored,...,NaN,people with attention deficit hyperactivity di...,fidgety | restless | easily bored,NaN,attention deficit hyperactivity disorder | att...,"[attention deficit hyperactivity disorder, att...",people with attention deficit hyperactivity di...,[people with attention deficit hyperactivity d...,attention deficit hyperactivity disorder | att...,"[attention deficit hyperactivity disorder, att..."
4,AIDA,acr,NaN,style,abbreviation,marketing,NaN,"attention, interest, desire, action | attentio...",NaN,NaN,...,NaN,NaN,NaN,NaN,"attention, interest, desire, action | attentio...","[attention, interest, desire, action, attentio...",,[],"attention, interest, desire, action | attentio...","[attention, interest, desire, action, attentio..."


### Create subsets of dataframes

In [107]:
df_rules = df_rules[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"]]

### Check duplicates&homonyms

In [108]:
df_rules[df_rules["Lemma"].duplicated(keep = False)]

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
532,challenge,v,unconscious_bias,agentic,"[question, cross-examine, re-evaluate, object ...",,[],"[question, cross-examine, re-evaluate, object ..."
533,challenge,s,unconscious_bias,agentic,"[test, burning question, complex situation, ta...",complex questions | difficult tasks | burning ...,"[complex questions, difficult tasks, burning q...","[test, burning question, complex situation, ta..."
673,direct,a,unconscious_bias,agentic,"[straightfoward, open, candid]",,[],"[straightfoward, open, candid]"
674,direct,v,unconscious_bias,agentic,"[guide, show, point, organize, encourage, assi...",,[],"[guide, show, point, organize, encourage, assi..."
714,drive,v,unconscious_bias,agentic,"[invest in, support, work toward, take respons...",,[],"[invest in, support, work toward, take respons..."
715,drive,s,unconscious_bias,agentic,"[spirit, energy, motivation, appetite, growth ...",,[],"[spirit, energy, motivation, appetite, growth ..."
727,elderly,s,unconscious_bias,age_old,"[older adult, older people]",older people,[older people],"[older adult, older people]"
728,elderly,a,unconscious_bias,age_old,[older],,[],[older]
787,female,a,unconscious_bias,gender_identity,"[--- only if gender identity is relevant, --- ...",--- only if gender identity is relevant | --- ...,"[--- only if gender identity is relevant, --- ...","[--- only if gender identity is relevant, --- ..."
788,female,s,unconscious_bias,gender_identity,"[person, adult, woman, you, they, team, crowd,...",team | crowd,"[team, crowd]","[person, adult, woman, you, they, person who s..."


### Homonyms

In [109]:
# check how many homonyms
df_rules["Lemma"].duplicated(keep = False).sum()

48

In [110]:
df_homonyms = df_rules[df_rules["Lemma"].duplicated(keep = False)]

In [111]:
#check first 5 rows
df_homonyms.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
532,challenge,v,unconscious_bias,agentic,"[question, cross-examine, re-evaluate, object ...",,[],"[question, cross-examine, re-evaluate, object ..."
533,challenge,s,unconscious_bias,agentic,"[test, burning question, complex situation, ta...",complex questions | difficult tasks | burning ...,"[complex questions, difficult tasks, burning q...","[test, burning question, complex situation, ta..."
673,direct,a,unconscious_bias,agentic,"[straightfoward, open, candid]",,[],"[straightfoward, open, candid]"
674,direct,v,unconscious_bias,agentic,"[guide, show, point, organize, encourage, assi...",,[],"[guide, show, point, organize, encourage, assi..."
714,drive,v,unconscious_bias,agentic,"[invest in, support, work toward, take respons...",,[],"[invest in, support, work toward, take respons..."


In [112]:
#create subset of dataframe to drop unnecessary columns
df_homonyms = df_homonyms[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [113]:
df_homonyms.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 48 entries, 532 to 1679
Data columns (total 5 columns):
Lemma                  48 non-null object
Word_Type              48 non-null object
Category               48 non-null object
Primary_subcategory    47 non-null object
Alt_split              48 non-null object
dtypes: object(5)
memory usage: 2.2+ KB


In [114]:
#save homonyms to csv file after preprocessing
df_homonyms.to_csv("homonyms_words_EN.csv", index=False)

### Remove homonyms from dataframe for the next steps pre-processing

In [115]:
homonyms_list = list(df_rules[df_rules["Lemma"].duplicated()]["Lemma"])

In [116]:
homonyms_list

['challenge',
 'direct',
 'drive',
 'elderly',
 'female',
 'fight',
 'force',
 'handicap',
 'impact',
 'lead',
 'male',
 'man',
 'master',
 'must-have',
 'musthave',
 'nag',
 'objective',
 'pick fights',
 'pioneer',
 'psycho',
 'queer',
 'sass',
 'schizo',
 'wacko']

In [117]:
df_rules = df_rules.query("Lemma != @homonyms_list")

### Acronyms (abbreviations)

In [118]:
#check word_type
df_rules["Word_Type"].unique()

array(['acr', 'abbr', 's', 'i', 'a+s', 'a', 'adv', 'v', 'v+a+s', 'v+s',
       'adv+a', 'v+a', 'a+a+s'], dtype=object)

In [119]:
df_abbr = df_rules[(df_rules["Word_Type"]=='abbr') | (df_rules["Word_Type"]=='acr')]

In [120]:
df_abbr.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 279 entries, 0 to 750
Data columns (total 8 columns):
Lemma                  279 non-null object
Word_Type              279 non-null object
Category               279 non-null object
Primary_subcategory    279 non-null object
Sg_all_split           279 non-null object
Pl_all                 279 non-null object
Pl_all_split           279 non-null object
Alt_split              279 non-null object
dtypes: object(8)
memory usage: 19.6+ KB


In [121]:
#create subset of dataframe to drop unnecessary columns
df_abbr = df_abbr[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [122]:
#save acronyms (abbreviations) to csv file after preprocessing
df_abbr.to_csv("abbreviations_EN.csv", index=False)

### Remove acronymes (abbr) from dataframe for the next steps pre-processing

In [123]:
#remove abbr
abbr_list = list(df_abbr["Lemma"])

In [124]:
df_rules = df_rules.query("Lemma != @abbr_list")

In [125]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1399 entries, 18 to 1726
Data columns (total 8 columns):
Lemma                  1399 non-null object
Word_Type              1399 non-null object
Category               1399 non-null object
Primary_subcategory    1399 non-null object
Sg_all_split           1399 non-null object
Pl_all                 1399 non-null object
Pl_all_split           1399 non-null object
Alt_split              1399 non-null object
dtypes: object(8)
memory usage: 98.4+ KB


### Openly_discriminating

In [126]:
df_openly_discrim = df_rules[df_rules["Category"]=="openly_discriminating"]

In [127]:
#info open_discrim
df_openly_discrim.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 342 entries, 18 to 1721
Data columns (total 8 columns):
Lemma                  342 non-null object
Word_Type              342 non-null object
Category               342 non-null object
Primary_subcategory    342 non-null object
Sg_all_split           342 non-null object
Pl_all                 342 non-null object
Pl_all_split           342 non-null object
Alt_split              342 non-null object
dtypes: object(8)
memory usage: 24.0+ KB


In [128]:
df_openly_discrim.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
18,African queen,s,openly_discriminating,homophobia,[-],-,[-],[-]
20,Aunt Jemima,s,openly_discriminating,racism,[-],-,[-],[-]
30,Barbie,s,openly_discriminating,racism,[-],-,[-],[-]
31,Beastie Boy,s,openly_discriminating,antisemitism,[-],-,[-],[-]
32,Beulah,s,openly_discriminating,racism,[-],-,[-],[-]


In [129]:
#split dataframe into two words and sentences using function split_words_sentences
df_open_dis_word, df_open_dis_sentence = split_words_sentences(df_openly_discrim)

In [130]:
df_open_dis_word.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
0,Barbie,s,openly_discriminating,racism,[-],-,[-],[-]
1,Beulah,s,openly_discriminating,racism,[-],-,[-],[-]
2,Bin-Laden,s,openly_discriminating,antimuslim,[-],-,[-],[-]
3,Binladen,s,openly_discriminating,antimuslim,[-],-,[-],[-]
4,Brownie,s,openly_discriminating,homophobia,[-],-,[-],[-]


In [131]:
#create subset of dataframe to drop unnecessary columns
df_open_dis_word = df_open_dis_word[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]
df_open_dis_sentence = df_open_dis_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [132]:
#save to csv file after preprocessing
df_open_dis_word.to_csv("open_dis_words_EN.csv", index=False)

In [133]:
#save to csv file after preprocessing
df_open_dis_sentence.to_csv("open_dis_sentences_EN.csv", index=False)

### Style

In [134]:
df_style = df_rules[df_rules["Category"]=="style"]

In [135]:
#info style
df_style.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 139 entries, 159 to 1718
Data columns (total 8 columns):
Lemma                  139 non-null object
Word_Type              139 non-null object
Category               139 non-null object
Primary_subcategory    139 non-null object
Sg_all_split           139 non-null object
Pl_all                 139 non-null object
Pl_all_split           139 non-null object
Alt_split              139 non-null object
dtypes: object(8)
memory usage: 9.8+ KB


In [136]:
#split dataframe into two words and sentences using function split_words_sentences
df_style_word, df_style_sentence = split_words_sentences(df_style)

In [137]:
#create subset of dataframe to drop unnecessary columns
df_style_word = df_style_word[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]
df_style_sentence = df_style_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [138]:
#save to csv file after preprocessing
df_style_word.to_csv("style_words_EN.csv", index=False)

In [139]:
#save to csv file after preprocessing
df_style_sentence.to_csv("style_sentences_EN.csv", index=False)

### Inclusive

In [140]:
df_inclusive = df_rules[df_rules["Category"]=="inclusive"]

In [141]:
#info inclusive
df_inclusive.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 185 entries, 330 to 1722
Data columns (total 8 columns):
Lemma                  185 non-null object
Word_Type              185 non-null object
Category               185 non-null object
Primary_subcategory    185 non-null object
Sg_all_split           185 non-null object
Pl_all                 185 non-null object
Pl_all_split           185 non-null object
Alt_split              185 non-null object
dtypes: object(8)
memory usage: 13.0+ KB


In [142]:
df_inclusive.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
330,accessibility,s,inclusive,d_and_i,[],,[],[]
337,affectionate,a,inclusive,communal,[],,[],[]
338,affinity group,s,inclusive,d_and_i,[],,[],[]
344,ages,s,inclusive,d_and_i,[],,[],[]
347,agree,v,inclusive,communal,[],,[],[]


In [143]:
#split dataframe into two words and sentences using function split_words_sentences
df_inclusive_word, df_inclusive_sentence = split_words_sentences(df_inclusive)

In [144]:
#create subset of dataframe to drop unnecessary columns
df_inclusive_word = df_inclusive_word[['Lemma', "Word_Type", "Category", "Primary_subcategory"]]
df_inclusive_sentence = df_inclusive_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory"]]

In [145]:
df_inclusive_sentence.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 4 columns):
Lemma                  24 non-null object
Word_Type              24 non-null object
Category               24 non-null object
Primary_subcategory    24 non-null object
dtypes: object(4)
memory usage: 896.0+ bytes


In [146]:
#save to csv file after preprocessing
df_inclusive_word.to_csv("inclusive_words_EN.csv", index=False)

In [147]:
#save to csv file after preprocessing
df_inclusive_sentence.to_csv("inclusive_sentences_EN.csv", index=False)

### Gendered

In [148]:
df_gendered = df_rules[df_rules["Category"]=="gendered"]

In [149]:
#info gendered
df_gendered.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 181 entries, 74 to 1694
Data columns (total 8 columns):
Lemma                  181 non-null object
Word_Type              181 non-null object
Category               181 non-null object
Primary_subcategory    181 non-null object
Sg_all_split           181 non-null object
Pl_all                 181 non-null object
Pl_all_split           181 non-null object
Alt_split              181 non-null object
dtypes: object(8)
memory usage: 12.7+ KB


In [150]:
#split dataframe into two words and sentences using function split_words_sentences
df_gendered_word, df_gendered_sentence = split_words_sentences(df_gendered)

In [151]:
df_gendered_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129 entries, 0 to 128
Data columns (total 8 columns):
Lemma                  129 non-null object
Word_Type              129 non-null object
Category               129 non-null object
Primary_subcategory    129 non-null object
Sg_all_split           129 non-null object
Pl_all                 129 non-null object
Pl_all_split           129 non-null object
Alt_split              129 non-null object
dtypes: object(8)
memory usage: 8.2+ KB


In [152]:
df_gendered_word_noun = df_gendered_word.query("Word_Type == 's'")

In [153]:
df_gendered_word_noun.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 113 entries, 0 to 127
Data columns (total 8 columns):
Lemma                  113 non-null object
Word_Type              113 non-null object
Category               113 non-null object
Primary_subcategory    113 non-null object
Sg_all_split           113 non-null object
Pl_all                 113 non-null object
Pl_all_split           113 non-null object
Alt_split              113 non-null object
dtypes: object(8)
memory usage: 7.9+ KB


In [154]:
df_gendered_word_no_noun = df_gendered_word.query("Word_Type != 's'")

In [155]:
#drop unnecessary column "Pl_all"
df_gendered_word_noun = df_gendered_word_noun.drop(labels=["Pl_all"], axis=1)
df_gendered_word_no_noun = df_gendered_word_no_noun.drop(labels=["Pl_all"], axis=1)
df_gendered_sentence = df_gendered_sentence.drop(labels=["Pl_all"], axis=1)

In [156]:
#save to csv file after preprocessing
df_gendered_word_noun.to_csv("gendered_noun_words_EN.csv", index=False)

In [157]:
#save to csv file after preprocessing
df_gendered_word_no_noun.to_csv("gendered_no_noun_words_EN.csv", index=False)

In [158]:
df_gendered_sentence.to_csv("gendered_sentences_EN.csv", index=False)

### Unconscious Bias

In [159]:
df_ub = df_rules[df_rules["Category"]=='unconscious_bias']

In [160]:
#info unc_bias
df_ub.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 552 entries, 19 to 1726
Data columns (total 8 columns):
Lemma                  552 non-null object
Word_Type              552 non-null object
Category               552 non-null object
Primary_subcategory    552 non-null object
Sg_all_split           552 non-null object
Pl_all                 552 non-null object
Pl_all_split           552 non-null object
Alt_split              552 non-null object
dtypes: object(8)
memory usage: 38.8+ KB


In [161]:
#split dataframe into two words and sentences using function split_words_sentences
df_ub_word, df_ub_sentence = split_words_sentences(df_ub)

In [162]:
df_ub_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 337 entries, 0 to 336
Data columns (total 8 columns):
Lemma                  337 non-null object
Word_Type              337 non-null object
Category               337 non-null object
Primary_subcategory    337 non-null object
Sg_all_split           337 non-null object
Pl_all                 337 non-null object
Pl_all_split           337 non-null object
Alt_split              337 non-null object
dtypes: object(8)
memory usage: 21.2+ KB


In [163]:
#filter datframe to have plural for subj(noun) without special caracters and empty strings
df_ub_word_pl = df_ub_word[(df_ub_word["Pl_all"]!='')&(df_ub_word["Pl_all"]!='-')&(df_ub_word["Word_Type"]=='s')]

In [164]:
df_ub_word_pl.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 93 entries, 0 to 329
Data columns (total 8 columns):
Lemma                  93 non-null object
Word_Type              93 non-null object
Category               93 non-null object
Primary_subcategory    93 non-null object
Sg_all_split           93 non-null object
Pl_all                 93 non-null object
Pl_all_split           93 non-null object
Alt_split              93 non-null object
dtypes: object(8)
memory usage: 6.5+ KB


#### save the left data into separate dataframe

In [165]:
plural_list = list(df_ub_word_pl["Lemma"])

In [166]:
df_ub_word_left = df_ub_word.query("Lemma != @plural_list")

#### remove unnecessary column "Pl_all" 

In [167]:
df_ub_word_left = df_ub_word_left.drop(labels=["Pl_all"], axis=1)
df_ub_word_pl = df_ub_word_pl.drop(labels=["Pl_all"], axis=1)
df_ub_sentence = df_ub_sentence.drop(labels=["Pl_all"], axis=1)

#### save to csv files after preprocessing

In [168]:
#save to csv files after preprocessing
df_ub_word_left.to_csv("ub_no_plur_words_EN.csv", index=False)
df_ub_word_pl.to_csv("ub_plur_words_EN.csv", index=False)
df_ub_sentence.to_csv("ub_sentences_EN.csv", index=False)